# Project

## Scrapping the data

In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver

In [24]:
driver = webdriver.Chrome()

driver.get("""https://www.transfermarkt.com/spieler-statistik/wertvollstespieler/marktwertetop?land_id=0
&ausrichtung=alle&spielerposition_id=alle&altersklasse=alle&jahrgang=0&kontinent_id=0&jahr=&only_loans=&plus=1""")

In [25]:
soup = BeautifulSoup(driver.page_source, "html.parser")

players = soup.select('table.items > tbody > tr')[:20]

data = []

for player in players:
    name = player.select_one('td.hauptlink a')

    cells = player.select('td')

    nationality = player.select_one('img.flaggenrahmen')

    club = player.select_one('td.zentriert a img')

    data.append({
        'Player': name.get_text(strip=True) if name else None,
        'Age': cells[5].get_text(strip=True) if len(cells) > 5 else None,
        'Position': cells[4].get_text(strip=True) if len(cells) > 4 else None,
        'Nationality': nationality.get('title') if nationality else None,
        'Club': club.get('alt') if club else None,
        'Market Value': player.select_one('td.rechts.hauptlink').get_text(strip=True)
                        if player.select_one('td.rechts.hauptlink') else None,
        'Matches': cells[9].get_text(strip=True) if len(cells) > 9 else None,
        'Goals': cells[10].get_text(strip=True) if len(cells) > 10 else None,
        'Assists': cells[12].get_text(strip=True) if len(cells) > 12 else None,
        'Yellow Cards': cells[13].get_text(strip=True) if len(cells) > 13 else None,
        'Red Cards': cells[15].get_text(strip=True) if len(cells) > 15 else None
    })

df = pd.DataFrame(data)

df.head(20)

,Player,Age,Position,Nationality,Club,Market Value,Matches,Goals,Assists,Yellow Cards,Red Cards
0,Lamine Yamal,19,Right Winger,Spain,FC Barcelona,€220.00m,37,25,17,5,0
1,Erling Haaland,26,Centre-Forward,Norway,Manchester City,€220.00m,39,26,5,3,0
2,Kylian Mbappé,27,Centre-Forward,France,Real Madrid,€200.00m,32,25,5,4,0
3,Michael Olise,24,Right Winger,France,FC Bayern Munich,€180.00m,37,21,22,7,0
4,Jude Bellingham,23,Attacking Midfield,England,Real Madrid,€160.00m,32,6,5,5,0
5,Pedri,23,Central Midfield,Spain,FC Barcelona,€150.00m,36,1,8,1,0
6,João Neves,22,Central Midfield,Portugal,Paris Saint-Germain,€140.00m,32,1,3,1,0
7,Khvicha Kvaratskhelia,25,Left Winger,Georgia,Paris Saint-Germain,€140.00m,38,15,7,3,0
8,Vitinha,26,Defensive Midfield,Portugal,Paris Saint-Germain,€140.00m,39,5,3,2,0
9,Vinicius Junior,26,Left Winger,Brazil,Real Madrid,€130.00m,40,18,8,9,1


## Cleaning the data

In [33]:
df.isnull().sum()

Player          0
Age             0
Position        0
Nationality     0
Club            0
Market Value    0
Matches         0
Goals           0
Assists         0
Yellow Cards    0
Red Cards       0
dtype: int64

In [28]:
df['Market Value'] = df['Market Value'].str.replace('€', '', regex=False)

df['Market Value'] = df['Market Value'].apply(
    lambda x: float(x.replace('m', '')) if 'm' in x
    else float(x.replace('k', '')) / 1000 if 'k' in x
    else None
)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Player        20 non-null     object 
 1   Age           20 non-null     object 
 2   Position      20 non-null     object 
 3   Nationality   20 non-null     object 
 4   Club          20 non-null     object 
 5   Market Value  20 non-null     float64
 6   Matches       20 non-null     object 
 7   Goals         20 non-null     object 
 8   Assists       20 non-null     object 
 9   Yellow Cards  20 non-null     object 
 10  Red Cards     20 non-null     object 
dtypes: float64(1), object(10)
memory usage: 1.8+ KB


In [31]:
df['Age'] = pd.to_numeric(df['Age'])
df['Matches'] = pd.to_numeric(df['Matches'])
df['Goals'] = pd.to_numeric(df['Goals'])
df['Assists'] = pd.to_numeric(df['Assists'])
df['Yellow Cards'] = pd.to_numeric(df['Yellow Cards'])
df['Red Cards'] = pd.to_numeric(df['Red Cards'])

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Player        20 non-null     object 
 1   Age           20 non-null     int64  
 2   Position      20 non-null     object 
 3   Nationality   20 non-null     object 
 4   Club          20 non-null     object 
 5   Market Value  20 non-null     float64
 6   Matches       20 non-null     int64  
 7   Goals         20 non-null     int64  
 8   Assists       20 non-null     int64  
 9   Yellow Cards  20 non-null     int64  
 10  Red Cards     20 non-null     int64  
dtypes: float64(1), int64(6), object(4)
memory usage: 1.8+ KB


In [34]:
df.to_csv('transfermarkt_players.csv', index=False)

In [36]:
pd.read_csv('transfermarkt_players.csv').head()

,Player,Age,Position,Nationality,Club,Market Value,Matches,Goals,Assists,Yellow Cards,Red Cards
0,Lamine Yamal,19,Right Winger,Spain,FC Barcelona,220.0,37,25,17,5,0
1,Erling Haaland,26,Centre-Forward,Norway,Manchester City,220.0,39,26,5,3,0
2,Kylian Mbappé,27,Centre-Forward,France,Real Madrid,200.0,32,25,5,4,0
3,Michael Olise,24,Right Winger,France,FC Bayern Munich,180.0,37,21,22,7,0
4,Jude Bellingham,23,Attacking Midfield,England,Real Madrid,160.0,32,6,5,5,0
